# AUTHOR: KEITH AARON R. NAVAL
# DATE: OCT 21, 2026

## PROBLEM:
A provider is considered to work at a clinic if they have the same physical location and (phone number or fax number or clinic name)

For the output, give me back these CSV, with:
A new column with a numeric ID for clinics

A new column Clinic ID, that references the most likely clinic the provider works at, for providers.

Use the clinic_name data from the provider sheet, to populate missing clinic names based on the associated ID.

With the leftover unmatched providers, generate new clinics objects using their data including a clinic_name if provided. Use “Unnamed” if you do not have a clinic_name available.

Also, send me your source code.

Also, provide a summary of what methods you used for linking, and how may providers were successfully linked out of the total using your methods.

Indicate also how many were ambiguous, and may work at more than one clinic.

Indicate how many could not be detected to work at any clinic so a new clinic was generated.

## ANSWER:
Total matched providers: 47
Total ambiguous records: 393
Total new clinics generated: 3527

Load CSV and Libraries

In [843]:
import pandas as pd
import numpy as np

clinic_df = pd.read_csv("vancouver-clinics.csv", header=None)
provider_df = pd.read_csv("college-vancouver-providers.csv", header=None, names=range(30))


vancouver-clinics.csv has two different formats. Format 1: (Row 0 to 2673) Format 2: (Row 2674 onwards)
If I cut columns, it'll delete important details such as the Address. As such, it is better to split the csv into two halves.


In [844]:
#Split Clinic DF into Two
clinic_top_half = clinic_df.iloc[:2673]
clinic_bottom_half = clinic_df.iloc[2673:]

Here, we remove unused columns that are hard to parse and keep only necessary columns.

In this task, I chose:
1. Clinic Name
2. Latitude
3. Longitude
4. Address

In [845]:
#Remove Clinic DF Top Half Unused Columns
#Keep only column 1 (Office Name), 3 (Latitude), 4 (Longitude), 5 (Address)
clinic_top_half = clinic_top_half.iloc[:,[1,3,4,5]]
clinic_top_half.columns = ["Clinic_Name", "Latitude", "Longitude", "Address"]
clinic_top_half

,Clinic_Name,Latitude,Longitude,Address
0,Unity Septic Svc,52.4437666,-109.1583812,"Farm, Unity, Saskatchewan, Canada"
1,Sayward Primary Clinic,50.3791228,-125.9597917,"601 Kelsey Way, Sayward, British Columbia, Canada"
2,Sayward Village,50.3791264,-125.9597364,"601 Kelsey Way, Sayward, British Columbia, Canada"
3,Tahsis Health Centre,49.9136623,-126.6632962,"1085 S Maquinna Dr, Tahsis, British Columbia, ..."
4,British Columbia Ambulance Service Station 135,49.9894121,-126.8519643,"500 Parkway Rd, Zeballos, British Columbia, Ca..."
...,...,...,...,...
2668,Fraser Alan B M D Ltd,49.246415,-124.7989448,"3855 9th Ave, Port Alberni, British Columbia, ..."
2669,"Dr. Jeannie Doig, Naturopathic Doctor",49.2607752,-124.8107515,"4711 Elizabeth St, Port Alberni, British Colum..."
2670,SpeechEase,49.2596667,-124.8071732,"Basement Suite, 4590 Helen St, Port Alberni, B..."
2671,Lund R Daryl Dr,49.246415,-124.7989448,"3855 9th Ave, Port Alberni, British Columbia, ..."


In [846]:
#Remove Clinic DF Bottom Half Unused Columns
clinic_bottom_half.head()
clinic_bottom_half = clinic_bottom_half.iloc[:,[0,6,7,9,12,13]]
clinic_bottom_half.columns = ["Clinic_Name", "Phone", "Fax", "Address", "Longitude", "Latitude"]
clinic_bottom_half

,Clinic_Name,Phone,Fax,Address,Longitude,Latitude
2673,Abbotsford Urgent and Primary Care Centre,6048703325,6048521351,"2692 Clearbrook Rd #100, Abbotsford, BC, Canada",49.0508301,-122.3370856
2674,Chilliwack Urgent and Primary Care Centre,7787046006,6047920573,"7955 Evans Road, Chilliwack, BC, Canada",49.146095,-121.97773
2675,Edmonds Urgent and Primary Care Centre,6045193787,6045193765,"201-7315 Edmonds Street, Burnaby, BC, Canada",49.218922,-122.950988
2676,Langley Urgent and Primary Care Centre,6045147477,7783661616,"101-20651 56th Avenue, Langley, BC, Canada",49.10503,-122.65052
2677,Metrotown Urgent and Primary Care Centre,6044514888,6044370149,"102-4555 Kingsway, Burnaby, BC, Canada",49.2298155,-123.002792
...,...,...,...,...,...,...
6555,West Kelowna Medical Clinic,2507686985,NaN,"105-2231 Louie Drive, West Kelowna, BC, Canada",49.837951,-119.615581
6556,West Saanich Medical Clinic,2505908378,NaN,"105-4442 W Saanich Road, Victoria, BC, Canada",48.4920083,-123.388236
6557,Whistler 360 Health,6049323977,NaN,"201-4380 Lorimer Road, Whistler, BC, Canada",50.119005,-122.954771
6558,Willoughby Medical Centre,6045148800,NaN,"20202 66th Avenue, Langley, BC, Canada",49.121094,-122.662801


After cleaning, we then concatenate the two halves and start cleaning the data. Since both halves have the same columns, concatenating and cleaning them won't be an issue.

In [847]:
#Concatenate and Clean Clinic DF Top and Bottom Half
clinic_df = pd.concat([clinic_top_half, clinic_bottom_half], ignore_index=True)

#Clean After Concatenating
clinic_df["Latitude"] = pd.to_numeric(clinic_df["Latitude"], errors="coerce").round(4) #Round Latitude to 4
clinic_df["Longitude"] = pd.to_numeric(clinic_df["Longitude"], errors="coerce").round(4) #Round Longitude to 4
clinic_df["Clinic_Name"] = clinic_df["Clinic_Name"].str.strip().str.lower() #Make clinic names lowercase
clinic_df = clinic_df.drop_duplicates(subset = ["Latitude", "Longitude", "Clinic_Name"]).reset_index(drop=True) #Remove Duplicate Clinics and Reset Index
clinic_df = clinic_df.dropna(subset=["Latitude", "Longitude"]) #Drop NAs
clinic_df = clinic_df.fillna("") #Fill NAs with empty
clinic_df["Clinic_ID"] = clinic_df.index + 1 #Create Clinic IDs
clinic_df.head()

,Clinic_Name,Latitude,Longitude,Address,Phone,Fax,Clinic_ID
0,unity septic svc,52.4438,-109.1584,"Farm, Unity, Saskatchewan, Canada",,,1
1,sayward primary clinic,50.3791,-125.9598,"601 Kelsey Way, Sayward, British Columbia, Canada",,,2
2,sayward village,50.3791,-125.9597,"601 Kelsey Way, Sayward, British Columbia, Canada",,,3
3,tahsis health centre,49.9137,-126.6633,"1085 S Maquinna Dr, Tahsis, British Columbia, ...",,,4
4,british columbia ambulance service station 135,49.9894,-126.8520,"500 Parkway Rd, Zeballos, British Columbia, Ca...",,,5


For the college-vancouver-providers.csv, splitting won't be necessary; however, there is unstructured or inconsistent data from rows 160 to 263. Removing them using strip() or replace() is hard because there are whitespaces and special symbols in the way. There are also entries that are mixed in, making row selection and deletion not possible. First thing I did was just ignore the consistent rows and dropped the unnecessary columns just like the clinic. In this case:

1. Phone
2. Fax
3. Provider Name
4. Latitude
5. Longitude
6. Clinic Name

In [848]:
#Provider DF Cleaning
provider_df = provider_df.iloc[:,[1,2,18,20,21,22]] #Locate Important Columns
provider_df.columns = ["Phone", "Fax", "Provider_Name", "Latitude", "Longitude", "Clinic_Name"]
provider_df["Latitude"] = pd.to_numeric(provider_df["Latitude"], errors="coerce").round(4) #Round Latitude to 4
provider_df["Longitude"] = pd.to_numeric(provider_df["Longitude"], errors="coerce").round(4) #Round Longitude to 4
provider_df["Provider_Name"] = provider_df["Provider_Name"].str.strip().str.lower() #format to lowercase
provider_df["Clinic_Name"] = provider_df["Clinic_Name"].str.strip().str.lower() #format to lowercase
provider_df = provider_df.drop_duplicates(subset = ["Latitude", "Longitude", "Provider_Name"]).reset_index(drop=True) #Remove Duplicate Providers and Reset Index
provider_df = provider_df.dropna(subset=["Latitude", "Longitude"]) #Drop NAs
provider_df = provider_df.fillna("") #Fill NAs with empty
provider_df["Clinic_ID"] = 0 #Create empty column
provider_df.head()

,Phone,Fax,Provider_Name,Latitude,Longitude,Clinic_Name,Clinic_ID
0,,,dr sathish rajasekaran,49.8567,-119.4895,,0
1,,,dr christopher adam robinson,48.4514,-123.3059,,0
2,,,dr john clifford saliken,49.2459,-124.7875,,0
3,,,dr dennis joseph small,49.1035,-117.7013,,0
4,,,dr zachary tan,49.0291,-122.8037,,0


Task 2.	A new column Clinic ID, that references the most likely clinic the provider works at, for providers.

Once both clinic and provider csv files are clean, it was time to merge them. I used a left merge for this because provider csv was bigger in volume. It was also easier to map clinics to providers than vice versa. Left/X = Provider. Right/Y = Clinic

In [849]:
#Merge Clinic to Provider DF from latitude and longitude (Left Merge) then find matches
#X - provider, #Y - clinic

merged_df = provider_df.merge(clinic_df, on = ["Latitude", "Longitude"], how="left") #Left Merge Command

###
#For the conditional statements, I checked if the phone/fax/clinic names matched on each csv file. It also important to make sure that it isn't blank.
phone_match = (merged_df["Phone_x"] == merged_df["Phone_y"]) & (merged_df["Phone_x"] != "")
fax_match = (merged_df["Fax_x"] == merged_df["Fax_y"]) & (merged_df["Fax_x"] != "")
clinic_match = (merged_df["Clinic_Name_x"] == merged_df["Clinic_Name_y"]) & (merged_df["Clinic_Name_x"] != "")
###

#Variable storing all 3
matches = phone_match | fax_match | clinic_match

print(f"Total matched providers: {matches.sum()}") #Print total matched
merged_df["Final_Clinic_ID"] = np.where(matches, merged_df["Clinic_ID_y"], 0) #Replace Final Clinic ID. If it was a match, replace it with the Clinic ID from the Clinic CSV, else 0
final_provider_df = merged_df[["Final_Clinic_ID", "Provider_Name", "Phone_x", "Fax_x", "Latitude", "Longitude", "Clinic_Name_x"]] #Create df storing updated clinic IDs and created columns.
final_provider_df.columns = ["Clinic_ID", "Name", "Phone", "Fax", "Latitude", "Longitude", "Clinic_Name"]
final_provider_df["Clinic_ID"] = final_provider_df["Clinic_ID"].astype(int) #Convert Clinic ID to integer (idk why but they keep turning to float)
ambiguous = final_provider_df.duplicated(subset=["Name", "Phone", "Fax"],keep=False).sum() #Count ambiguous records
print(f"Total ambiguous records: {ambiguous}")
final_provider_df = final_provider_df.drop_duplicates(subset=["Name", "Phone", "Fax"]) #Drop duplicate phone/fax/names
final_provider_df.to_csv("provider_final.csv", index=False) #Output final csv



Total matched providers: 47
Total ambiguous records: 393


Task 3. Use the clinic_name data from the provider sheet, to populate missing clinic names based on the associated ID. First, create a condition checking if entries match from last time (matches), Clinic Name from clinic csv is empty, and Clinic Name from provider.csv is not empty,

In [850]:
no_clinic_matches = matches & (merged_df["Clinic_Name_y"] == "") & ((merged_df["Clinic_Name_x"]).str.len() != 0) #must be a match, clinic csv has no name for this clinic, provider has a name for it
#Create dictionary mapping Clinic_ID to Clinic_Name
clinic_dict = merged_df[no_clinic_matches].set_index("Clinic_ID_y")["Clinic_Name_x"].to_dict() #Clinic ID is key, Clinic Name is value
clinic_df["Clinic_Name"] = clinic_df["Clinic_ID"].map(clinic_dict).fillna(clinic_df["Clinic_Name"]) #Applies to clinic_df

Task 4.	With the leftover unmatched providers, generate new clinics objects using their data including a clinic_name if provided. Use “Unnamed” if you do not have a clinic_name available.



In [853]:
leftover_df = final_provider_df[final_provider_df["Clinic_ID"] == 0] #Checks for leftover clinics which wasn't assigned an ID.
leftover_df["Clinic_Name"] = leftover_df["Clinic_Name"].replace("", "Unnamed") #Replace Empty with Unnamed
leftover_df = leftover_df[["Clinic_Name", "Phone", "Fax", "Latitude", "Longitude"]].drop_duplicates(subset=["Latitude", "Longitude"])
clinic_df = pd.concat([clinic_df,leftover_df],ignore_index=True) #Concatenate clinic df and leftover df
clinic_df["Clinic_ID"] = clinic_df.index + 1 #Create clinic ID for leftovers
clinic_df.to_csv("clinic_final.csv",index=False)
clinic_df
print(f"Total new clinics generated: {len(leftover_df)}")

Total new clinics generated: 3527
